<a href="https://colab.research.google.com/github/deepan98raj-dotcom/Assessment/blob/main/RAG_with_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. Install Required Dependencies
# ==========================================
!pip install -q -U langchain langchain-community langchain-groq chromadb pypdf sentence-transformers langchain-huggingface langchain-text-splitters gradio requests

# ==========================================
# 2. Imports & Configuration
# ==========================================
import os
import warnings
import requests
from google.colab import userdata

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
import gradio as gr

warnings.filterwarnings('ignore')

# Safely load Groq API key from Colab Secrets
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

print("Environment setup complete.")

# ==========================================
# 3. Download & Load PDF Document
# ==========================================
pdf_url = "https://mml-book.github.io/book/mml-book.pdf"
pdf_path = "mathematics-for-machine-learning.pdf"

# Download PDF securely with browser user-agent
response = requests.get(pdf_url, headers={'User-Agent': 'Mozilla/5.0'})
with open(pdf_path, 'wb') as f:
    f.write(response.content)

loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages.")

# ==========================================
# 4. Text Chunking
# ==========================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
    separators=['\n\n', '\n', ' ', '']
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} text chunks.")

# ==========================================
# 5. Embeddings & Vector Store Setup
# ==========================================
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db_ml",
    collection_name="mathematics-for-machine-learning"
)

print(f"Vector store created with {vector_db._collection.count()} chunks.")

# ==========================================
# 6. LLM Initialization (Groq)
# ==========================================
llm = ChatGroq(
    groq_api_key=os.environ['GROQ_API_KEY'],
    model_name='llama-3.1-8b-instant',
    temperature=0.0,
    max_tokens=1000
)

# ==========================================
# 7. Prompt Builder Function
# ==========================================
def build_augmented_prompt(query, retrieved_docs):
    context = "\n\n".join([f"[Source Page {doc.metadata.get('page', 'N/A')}]:\n{doc.page_content}" for doc in retrieved_docs])

    prompt = f"""You are a helpful assistant answering questions about the Mathematics for Machine Learning textbook.

Answer the question strictly using ONLY the context provided below. If the answer cannot be found in the context, state "I cannot find sufficient information in the provided document."

Context:
{context}

Question: {query}
Answer:"""
    return prompt

# ==========================================
# 8. Gradio UI Application
# ==========================================
def rag_answer(message, history):
    # Retrieve top 4 relevant context chunks
    retrieved_docs = vector_db.similarity_search(message, k=4)

    # Build prompt
    prompt = build_augmented_prompt(message, retrieved_docs)

    # Query LLM
    response = llm.invoke(prompt)
    return response.content

demo = gr.ChatInterface(
    fn=rag_answer,
    title="Mathematics for ML — RAG Q&A",
    description="Ask any math or machine learning question based on the textbook.",
    examples=[
        "What are the core mathematical concepts required for machine learning?",
        "How is differentiation used in training neural networks?",
        "Why is integration important for uncertainty propagation in ML models?"
    ]
)

demo.launch(share=True, debug=True)

Environment setup complete.


Loaded 417 pages.
Created 1430 text chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created with 2860 chunks.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c1014ed789286ebcf8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 861, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2695, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/utils.py", line 1083, in async_wrapper
    response = await 